# 第 42 课：从卷积与注意力到 Conformer

这一课回答四个问题：为什么纯卷积和纯 Transformer 都不够理想？Conformer 怎样同时建模局部发音和长距离上下文？`[B,T,D]` 的 shape 怎样流动？流式 Conformer 还需要改什么？

建议先完成第 7～14 课。CPU 约需 1～3 分钟。

## 学习导航与完成标准

完成后你应能：

1. 解释卷积、注意力、前馈网络各自负责什么；
2. 写出 Conformer block 的残差顺序；
3. 正确构造 padding mask，并说明 mask 为什么不改变时间长度；
4. 证明全上下文注意力会读取未来；
5. 说出改成流式模型必须处理的 attention window、因果卷积与 cache。

证据门槛：闭卷画出 block，从空白补完 `ConformerBlock.forward`，并通过本课三个断言。

## 课前诊断（不要运行代码）

1. 一个音素主要是局部模式，还是需要整句话才能辨认？
2. “我想吃苹___”中的缺失字为什么可能需要较远的上下文？
3. 输入 `[B,T,D]=[2,100,80]` 经过不下采样的编码块，输出 shape 应是什么？

先写下答案。第 1、2 题并不矛盾：现代编码器正是要同时处理两种尺度。

<!-- course-bridge-v3 -->
## 知识接力：先取回旧知识，再进入本课

### 3 分钟闭卷回忆

在新 Markdown cell 中回答，**不要先翻前文**：CTC/流式状态和延迟；训练/测试数据权限；基线、消融与外部评测证据。

- 三项都能用“含义 + 单位/shape + 一个数字例子”回答：进入本课。
- 能回答两项：学习本课，但把缺口记入 `LEARNING_LOG.md`。
- 只能回答零到一项：先回到 [上一课](41_LLM语义后处理与端到端语音系统.ipynb)与[唯一学习路径](../LEARNING_PATH.md)，做一次最小实验；不要靠继续看新术语掩盖断点。

### 本课接口契约

```text
输入：明确任务、数据、算力、延迟与风险约束
  ↓ 本课要学会的变换、状态或判断
输出：能与 CTC/RNN-T/AED/LALM 基线公平比较的现代模型实验
```

学完后必须能解释：输入的哪个单位/shape/状态若丢失，会让输出“仍能运行却语义错误”。


In [ ]:
import math
import random
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(7)
np.random.seed(7)
random.seed(7)
torch.set_num_threads(2)

print("torch:", torch.__version__)
print("device: cpu（本课故意保持小模型，CPU 即可）")

## 1. 为什么是 Conformer？

语音同时有两种结构：

- **局部结构**：几十毫秒内的共振峰、爆破、摩擦等发音线索，卷积擅长；
- **全局结构**：较远词语、句法和说话上下文，注意力擅长。

Conformer 把两者放在同一个残差块中。常见的 Macaron 形式为：

$$
x_1=x+\tfrac12\mathrm{FFN}(x)
$$
$$
x_2=x_1+\mathrm{MHSA}(x_1)
$$
$$
x_3=x_2+\mathrm{Conv}(x_2)
$$
$$
y=\mathrm{LayerNorm}(x_3+\tfrac12\mathrm{FFN}(x_3))
$$

两个 FFN 各乘 $1/2$，注意力负责全局，卷积负责局部。残差连接让每个模块只需学习对当前表示的修正。

In [ ]:
def lengths_to_padding_mask(lengths: torch.Tensor, max_len=None):
    """返回 [B,T]；True 表示 padding，供 MultiheadAttention 使用。"""
    max_len = int(max_len or lengths.max())
    t = torch.arange(max_len, device=lengths.device)
    return t.unsqueeze(0) >= lengths.unsqueeze(1)

lengths = torch.tensor([7, 4])
padding_mask = lengths_to_padding_mask(lengths)
print(padding_mask.int())

expected = torch.tensor([
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 1, 1],
], dtype=torch.bool)
assert torch.equal(padding_mask, expected)
print("断言通过：True 只出现在第二条样本的 padding 区域。")

### Mask 最容易犯的错

`lengths=[7,4]` 表示两条序列的真实长度。PyTorch `MultiheadAttention` 的 `key_padding_mask=True` 表示“不要读取这里”，不是“这里有效”。不同 API 的布尔语义可能相反，必须查清契约。

Mask 不会删除时间步，因此输入输出仍是 `[B,T,D]`。为了防止 padding 位置经过残差后出现非零值，我们在 block 末尾再显式清零。

In [ ]:
class FeedForwardModule(nn.Module):
    def __init__(self, d_model, expansion=4, dropout=0.0):
        super().__init__()
        hidden = d_model * expansion
        self.net = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, hidden),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class ConvModule(nn.Module):
    """教学版 Conformer 卷积模块；保持 [B,T,D] 不变。"""
    def __init__(self, d_model, kernel_size=7, dropout=0.0):
        super().__init__()
        assert kernel_size % 2 == 1, "离线 same padding 需要奇数 kernel"
        self.norm = nn.LayerNorm(d_model)
        self.pointwise_in = nn.Conv1d(d_model, 2 * d_model, 1)
        self.depthwise = nn.Conv1d(
            d_model, d_model, kernel_size,
            padding=kernel_size // 2, groups=d_model,
        )
        self.batch_norm = nn.BatchNorm1d(d_model)
        self.pointwise_out = nn.Conv1d(d_model, d_model, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        z = self.norm(x).transpose(1, 2)       # [B,D,T]
        z = F.glu(self.pointwise_in(z), dim=1) # 2D -> D
        z = self.depthwise(z)                  # 每个通道独立做时间卷积
        z = F.silu(self.batch_norm(z))
        z = self.pointwise_out(z).transpose(1, 2)
        return self.dropout(z)

## 2. 组装一个 Conformer block

注意 `batch_first=True` 后，注意力输入是 `[B,T,D]`。如果漏掉它，代码可能不会立刻报错，却会把 batch 当成时间维，这是危险的静默错误。

In [ ]:
class ConformerBlock(nn.Module):
    def __init__(self, d_model=32, num_heads=4, kernel_size=7, dropout=0.0):
        super().__init__()
        self.ffn1 = FeedForwardModule(d_model, dropout=dropout)
        self.attn_norm = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            d_model, num_heads, dropout=dropout, batch_first=True
        )
        self.conv = ConvModule(d_model, kernel_size, dropout)
        self.ffn2 = FeedForwardModule(d_model, dropout=dropout)
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, x, padding_mask=None, attn_mask=None):
        x = x + 0.5 * self.ffn1(x)
        z = self.attn_norm(x)
        z, _ = self.attn(
            z, z, z,
            key_padding_mask=padding_mask,
            attn_mask=attn_mask,
            need_weights=False,
        )
        x = x + z
        x = x + self.conv(x)
        x = self.final_norm(x + 0.5 * self.ffn2(x))
        if padding_mask is not None:
            x = x.masked_fill(padding_mask.unsqueeze(-1), 0.0)
        return x


B, T, D = 2, 40, 32
x = torch.randn(B, T, D)
lengths = torch.tensor([40, 27])
mask = lengths_to_padding_mask(lengths, T)
block = ConformerBlock(d_model=D)
block.eval()

with torch.no_grad():
    y = block(x, padding_mask=mask)

print("input:", tuple(x.shape), "output:", tuple(y.shape))
print("padding 最大绝对值:", y[1, 27:].abs().max().item())
assert y.shape == x.shape
assert torch.count_nonzero(y[1, 27:]) == 0
print("断言通过：shape 保持不变，padding 输出为 0。")

## 3. 全上下文为什么不能直接流式？

离线 self-attention 中，第 5 帧可以读取第 50 帧。流式系统在第 5 帧到来时尚未收到第 50 帧，因此不能这样算。

下面只隔离注意力模块做实验：保留前半段不变，只大幅修改未来。如果过去的输出随之改变，就证明模型读取了未来。

In [ ]:
attn = nn.MultiheadAttention(16, 4, batch_first=True, dropout=0.0).eval()
base = torch.randn(1, 12, 16)
future_changed = base.clone()
future_changed[:, 6:] += 20.0

causal_mask = torch.triu(torch.ones(12, 12, dtype=torch.bool), diagonal=1)

with torch.no_grad():
    full_a, _ = attn(base, base, base, need_weights=False)
    full_b, _ = attn(future_changed, future_changed, future_changed, need_weights=False)
    causal_a, _ = attn(base, base, base, attn_mask=causal_mask, need_weights=False)
    causal_b, _ = attn(
        future_changed, future_changed, future_changed,
        attn_mask=causal_mask, need_weights=False,
    )

full_past_change = (full_a[:, :6] - full_b[:, :6]).abs().max().item()
causal_past_change = (causal_a[:, :6] - causal_b[:, :6]).abs().max().item()
print(f"全上下文：未来改变导致过去最大变化 {full_past_change:.6f}")
print(f"因果注意力：未来改变导致过去最大变化 {causal_past_change:.6f}")
assert full_past_change > 1e-3
assert causal_past_change < 1e-5
print("断言通过：因果 mask 阻止注意力读取未来。")

### 重要边界

给注意力加 causal mask **还不等于整个 Conformer 已经流式化**。上面的 `ConvModule` 使用左右对称 padding，仍会读取未来帧。真正的流式实现还需要：

1. 卷积只做左 padding，并跨 chunk 保存最近 `kernel_size-1` 帧；
2. 注意力限制左/右上下文，并缓存过去的 K/V；
3. 位置编码在 chunk 边界保持连续；
4. padding mask、cache 长度和下采样后的时间坐标一致；
5. 用“整段推理 vs 任意切块推理”一致性测试验收。

## 4. 接上 CTC 头

Conformer 是编码器，不直接规定训练目标。最简单的组合是在每个时间步接线性层，输出 `vocab_size + 1` 类 logits，再使用 CTC loss。这里用合成数据验证 shape 与梯度通路，不冒充真实识别准确率。

In [ ]:
class TinyConformerCTC(nn.Module):
    def __init__(self, feat_dim=24, d_model=32, vocab_size=8):
        super().__init__()
        self.input_proj = nn.Linear(feat_dim, d_model)
        self.encoder = nn.ModuleList([
            ConformerBlock(d_model, num_heads=4, kernel_size=7)
            for _ in range(2)
        ])
        self.ctc_head = nn.Linear(d_model, vocab_size + 1)  # 0 是 blank

    def forward(self, features, lengths):
        mask = lengths_to_padding_mask(lengths, features.size(1))
        x = self.input_proj(features)
        for layer in self.encoder:
            x = layer(x, padding_mask=mask)
        return self.ctc_head(x)


model = TinyConformerCTC()
features = torch.randn(2, 30, 24)
input_lengths = torch.tensor([30, 24], dtype=torch.long)
targets = torch.tensor([1, 2, 3, 4, 2, 5, 6], dtype=torch.long)
target_lengths = torch.tensor([4, 3], dtype=torch.long)

logits = model(features, input_lengths)              # [B,T,C]
log_probs = logits.log_softmax(-1).transpose(0, 1)   # CTCLoss 要 [T,B,C]
loss = nn.CTCLoss(blank=0, zero_infinity=True)(
    log_probs, targets, input_lengths, target_lengths
)
loss.backward()

grad_norm = model.input_proj.weight.grad.norm().item()
print("logits:", tuple(logits.shape))
print(f"CTC loss={loss.item():.4f}, input_proj grad norm={grad_norm:.4f}")
assert logits.shape == (2, 30, 9)
assert math.isfinite(loss.item()) and grad_norm > 0
print("断言通过：Conformer → CTC 的前向和反向链路完整。")

## 5. 分层练习

### A. 回忆与解释（每题 1 分）

1. 卷积和注意力分别擅长哪种时间尺度？
2. 为什么 Conformer 中有两个乘 $1/2$ 的 FFN？
3. `key_padding_mask=True` 在本课代码中表示有效还是无效？
4. Conformer 和 CTC 是同一层面的概念吗？

### B. 预测与计算（每题 2 分）

5. `[B,T,D]=[3,120,64]` 通过不下采样 block 后 shape 是什么？
6. kernel size 为 15 的对称卷积，单层每个位置最多读取左右各多少帧？
7. 如果前端每帧步长 10 ms、总下采样 4 倍，编码器 100 个时间步覆盖约多少秒？
8. 为什么 `target_length > input_length` 会使 CTC 无法对齐？

### C. 编程与排错（每题 3 分）

9. 删除 block 末尾的 `masked_fill`，观察 padding 输出并解释结果。
10. 故意去掉 `batch_first=True`，记录 shape 或语义错误。
11. 把 `kernel_size` 改为 15，统计参数量是否线性增加。
12. 从空白 cell 重写 `lengths_to_padding_mask` 和 block 的残差顺序。

满分 24；达到 19 分且能从空白画出结构，再进入 RNN-T/TDT。

## 离场小测（闭卷发给老师）

1. 用不超过 80 字解释 Conformer 为什么适合语音。
2. 写出输入 `[B,T,F]` 到 CTC logits 的 shape 变化。
3. 为什么“causal attention”不等于“流式 Conformer”？
4. 画出 FFN → MHSA → Conv → FFN 的残差结构。

请同时写出你最没有把握的一题。老师会依据错误类型决定补讲还是进入第 43 课。